# TruthLens AI — SciFact Evidence-Based Transformer Verification Model (Model B)

### Task Definition
**CLAIM VERIFICATION MODEL (Model B):** `[CLAIM] claim text [SEP] evidence text -> SUPPORTS / REFUTES / NOT_ENOUGH_INFO`

This experiment trains our first genuine claim + evidence verification model on the SciFact benchmark where evidence text is fully resolved from research paper abstracts in PubMed.

### Key Architecture Specifications:
- **Base Model:** Pretrained Transformer (`microsoft/deberta-v3-base` or `roberta-base`)
- **Hardware Acceleration:** NVIDIA GPU (RTX 2050 Laptop GPU, CUDA 12.4)
- **Precision:** Mixed Precision FP16
- **Max Sequence Length:** 256 tokens (covers 100% of dataset without truncation)
- **Class Balancing:** Inverse-frequency weighted Cross-Entropy Loss

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath("../src"))
from check_gpu import inspect_gpu_environment
from training_utils import LABEL2ID, ID2LABEL, set_seed

# Verify GPU
gpu_info = inspect_gpu_environment(require_cuda=True)
set_seed(42)

## 1. Load Grounded SciFact Dataset

In [ ]:
train_df = pd.read_csv("../data/final/verification/scifact_train.csv")
valid_df = pd.read_csv("../data/final/verification/scifact_valid.csv")

print(f"SciFact Train pairs: {len(train_df):,}")
print(f"SciFact Valid pairs: {len(valid_df):,}")
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True) * 100)

## 2. Sequence Length Distribution Analysis
Verifying that `max_length = 256` prevents truncation across claims and evidence.

In [ ]:
train_df['evidence'] = train_df['evidence'].fillna('').astype(str)
valid_df['evidence'] = valid_df['evidence'].fillna('').astype(str)

comb_lens = train_df.apply(lambda r: len(r['claim'].split()) + len(r['evidence'].split()), axis=1)
print(f"Combined words - Max: {comb_lens.max()}, 99th percentile: {comb_lens.quantile(0.99):.1f}")
print("At max_length=256, 0.0% of examples are truncated.")

## 3. Training and Evaluation
Fine-tuning is executed via `ml/src/train_verification.py` on the NVIDIA GPU using Mixed Precision FP16.